# Import 

In [ ]:
# Task: Tự xây dựng 1 mô hình CNN để phân loại 2-10 loại động vật (tùy chọn số lượng categories). 
# Dùng bất kì framework nào (pytorch, tensorflow, keras), không dùng mô hình có sẵn (resnet, vgg, alexnet, efficientnet,....)
# Chỉ train 1 vài epoch. Bài tập đánh giá cách xây dựng mô hình, quá trình huấn luyện, chứ không dựa vào accuracy

# Import libraries:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import random

# Define classes:
class AnimalDataset(Dataset):
    def __init__(self, root_dir, class_names, transform=None):
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}
        self.transform = transform

        for cls_name in class_names:
            cls_folder = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_folder):
                if fname.endswith((".jpg", ".png", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_folder, fname))
                    self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Transform & load data:
class_names = ['butterfly', 'cat', 'chicken', 'cow', 'dog', 
               'elephant', 'horse', 'sheep', 'spider', 'squirrel']

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

train_dataset = AnimalDataset("/Users/ngocta/Desktop/CoderSchool-AI/Week14/Train/train", class_names, transform = transform)
test_dataset = AnimalDataset("/Users/ngocta/Desktop/CoderSchool-AI/Week14/Test/test", class_names, transform = transform)
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

# Define CNN model:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding = 1)  
        self.conv2 = nn.Conv2d(16, 32, 3, padding = 1)
        self.pool = nn.MaxPool2d(2, 2)              
        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 16, 64, 64]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 32, 32, 32]
        
        # Flatten the tensor:
        x = x.view(-1, 32 * 32 * 32)         
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Train model:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes = len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

# Train with 5 epoch:
for epoch in range(5): 
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# Model evaluation:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {100 * correct / total:.2f}%")